# Splunk SPL — stats, eval, transaction, rex, timechart

This notebook is a **Senior Data Engineer field guide** to core Splunk SPL patterns using a Citi-style telemetry narrative.

## Mental model

Splunk is not just "search over logs." In operations, it acts like a **real-time operational intelligence engine**:

- **PostgreSQL** stores operational source data.
- **Splunk HEC** receives live event payloads.
- **SPL** transforms raw events into counts, scores, timelines, grouped incidents, and extracted fields.
- **REST APIs** let us automate search jobs and alert creation.

### What we are doing here

1. Read **500 real alert rows** from PostgreSQL.
2. Send them to **Splunk HEC** on `localhost:8088`.
3. Run **6 SPL queries** against Splunk REST API on `localhost:8089/services/search/jobs`.
4. Capture results back into pandas DataFrames.
5. Show how to think about **stats, eval, timechart, transaction, rex, and alert automation**.

### Citi framing

A Citi SOC / observability analyst may monitor **6,000+ endpoints** for latency, throughput, and error-rate degradation. These SPL commands are exactly the kinds of primitives used to triage operational events and escalate incidents fast.

In [ ]:
import os
import json
import time
import urllib3
from typing import Dict

import pandas as pd
import psycopg2
import requests
from requests.auth import HTTPBasicAuth
from IPython.display import display, Markdown

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

CONFIG = {
    "postgres": {
        "host": "localhost",
        "port": 5432,
        "dbname": "de_telemetry",
        "user": "de_admin",
        "password": "DeAdmin2026!",
    },
    "splunk": {
        "hec_url": "http://localhost:8088/services/collector",
        "hec_token": "f9d0f92a-fcad-4a02-a76e-0b9a325cffe8",
        "mgmt_base_url": "https://localhost:8089",
        "mgmt_verify_ssl": False,
        "index": "citi_telemetry",
        "source": "postgres.alerts",
        "sourcetype": "_json",
        "app": "search",
        "owner": "admin",
        "mgmt_username": os.getenv("SPLUNK_USERNAME"),
        "mgmt_password": os.getenv("SPLUNK_PASSWORD"),
        "mgmt_bearer_token": os.getenv("SPLUNK_BEARER_TOKEN"),
    },
    "dataset": {
        "hec_event_limit": 500,
    },
    "rest": {
        "poll_interval_seconds": 1.5,
        "poll_timeout_seconds": 60,
    }
}

if not (CONFIG["splunk"]["mgmt_bearer_token"] or (CONFIG["splunk"]["mgmt_username"] and CONFIG["splunk"]["mgmt_password"])):
    raise RuntimeError(
        "Splunk management credentials are required for search/jobs and saved searches. "
        "Set SPLUNK_USERNAME and SPLUNK_PASSWORD, or set SPLUNK_BEARER_TOKEN, then rerun."
    )

print("Configuration loaded.")
print(f"PostgreSQL target : {CONFIG['postgres']['host']}:{CONFIG['postgres']['port']}/{CONFIG['postgres']['dbname']}")
print(f"Splunk HEC        : {CONFIG['splunk']['hec_url']}")
print(f"Splunk Mgmt       : {CONFIG['splunk']['mgmt_base_url']}")
print(f"Splunk index      : {CONFIG['splunk']['index']}")

In [ ]:
def get_pg_connection():
    return psycopg2.connect(
        host=CONFIG["postgres"]["host"],
        port=CONFIG["postgres"]["port"],
        dbname=CONFIG["postgres"]["dbname"],
        user=CONFIG["postgres"]["user"],
        password=CONFIG["postgres"]["password"],
    )

def fetch_alert_events(limit: int = 500) -> pd.DataFrame:
    sql = f'''
    SELECT
        a.alert_id,
        a.endpoint_id,
        a.severity,
        a.message,
        a.created_at,
        e.name AS endpoint_name,
        e.region,
        e.status AS endpoint_status,
        e.category
    FROM alerts a
    JOIN endpoints e
      ON a.endpoint_id = e.endpoint_id
    ORDER BY a.created_at DESC, a.alert_id DESC
    LIMIT {int(limit)}
    '''
    with get_pg_connection() as conn:
        df = pd.read_sql_query(sql, conn)
    df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
    return df

alerts_df = fetch_alert_events(CONFIG["dataset"]["hec_event_limit"])
print(f"Fetched {len(alerts_df):,} alert rows from PostgreSQL.")
display(alerts_df.head(10))

## Setup — send 500 alert events to Splunk HEC

We are using **real alert rows from PostgreSQL**, not synthetic mock records.

Each event sent to Splunk includes:

- `alert_id`
- `endpoint_id`
- `severity`
- `message`
- `created_at`
- `endpoint_name`
- `region`
- `endpoint_status`
- `category`

That gives SPL enough structure to demonstrate operational analytics patterns.

In [ ]:
def build_hec_event(row: pd.Series) -> Dict:
    return {
        "time": row["created_at"].timestamp(),
        "host": "localhost",
        "source": CONFIG["splunk"]["source"],
        "sourcetype": CONFIG["splunk"]["sourcetype"],
        "index": CONFIG["splunk"]["index"],
        "event": {
            "alert_id": int(row["alert_id"]),
            "endpoint_id": int(row["endpoint_id"]),
            "severity": str(row["severity"]),
            "message": str(row["message"]),
            "created_at": row["created_at"].isoformat(),
            "endpoint_name": str(row["endpoint_name"]),
            "region": str(row["region"]),
            "endpoint_status": str(row["endpoint_status"]),
            "category": str(row["category"]),
            "pipeline_context": "citi_observability",
        },
    }

def send_events_to_splunk_hec(df: pd.DataFrame, batch_size: int = 100):
    headers = {
        "Authorization": f"Splunk {CONFIG['splunk']['hec_token']}",
        "Content-Type": "application/json",
    }
    sent = 0
    failed = 0
    records = [build_hec_event(row) for _, row in df.iterrows()]
    for i in range(0, len(records), batch_size):
        batch = records[i:i + batch_size]
        payload = "\n".join(json.dumps(item) for item in batch)
        response = requests.post(
            CONFIG["splunk"]["hec_url"],
            headers=headers,
            data=payload.encode("utf-8"),
            timeout=30,
        )
        if response.ok:
            sent += len(batch)
        else:
            failed += len(batch)
            raise RuntimeError(
                f"HEC send failed for batch starting at {i}: "
                f"status={response.status_code}, body={response.text}"
            )
    return {"sent": sent, "failed": failed}

hec_result = send_events_to_splunk_hec(alerts_df, batch_size=100)
print("HEC ingest result:", hec_result)

In [ ]:
class SplunkClient:
    def __init__(self, base_url: str, verify_ssl: bool = False):
        self.base_url = base_url.rstrip("/")
        self.verify_ssl = verify_ssl
        self.session = requests.Session()
        self.session.verify = verify_ssl

        bearer = CONFIG["splunk"]["mgmt_bearer_token"]
        user = CONFIG["splunk"]["mgmt_username"]
        password = CONFIG["splunk"]["mgmt_password"]

        if bearer:
            self.session.headers.update({"Authorization": f"Bearer {bearer}"})
        elif user and password:
            self.session.auth = HTTPBasicAuth(user, password)
        else:
            raise RuntimeError("No Splunk management credentials available.")

    def create_search_job(self, search: str, earliest_time: str = "-24h@h", latest_time: str = "now") -> str:
        url = f"{self.base_url}/services/search/jobs"
        payload = {
            "search": search if search.strip().startswith("search ") else f"search {search}",
            "earliest_time": earliest_time,
            "latest_time": latest_time,
            "output_mode": "json",
        }
        response = self.session.post(url, data=payload, timeout=30)
        response.raise_for_status()
        data = response.json()
        sid = data["sid"]
        return sid

    def wait_for_job(self, sid: str, timeout_seconds: int = 60, poll_interval_seconds: float = 1.5) -> None:
        url = f"{self.base_url}/services/search/jobs/{sid}"
        deadline = time.time() + timeout_seconds
        while time.time() < deadline:
            response = self.session.get(url, params={"output_mode": "json"}, timeout=30)
            response.raise_for_status()
            entry = response.json()["entry"][0]
            content = entry["content"]
            dispatch_state = content.get("dispatchState")
            is_done = content.get("isDone", False)
            if is_done or dispatch_state == "DONE":
                return
            time.sleep(poll_interval_seconds)
        raise TimeoutError(f"Timed out waiting for Splunk job {sid}.")

    def get_results(self, sid: str, count: int = 1000) -> pd.DataFrame:
        url = f"{self.base_url}/services/search/jobs/{sid}/results"
        response = self.session.get(
            url,
            params={"output_mode": "json", "count": count},
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        return pd.DataFrame(data.get("results", []))

    def run_search(self, search: str, earliest_time: str = "-24h@h", latest_time: str = "now", count: int = 1000) -> pd.DataFrame:
        sid = self.create_search_job(search=search, earliest_time=earliest_time, latest_time=latest_time)
        self.wait_for_job(
            sid=sid,
            timeout_seconds=CONFIG["rest"]["poll_timeout_seconds"],
            poll_interval_seconds=CONFIG["rest"]["poll_interval_seconds"],
        )
        return self.get_results(sid=sid, count=count)

    def create_saved_search(
        self,
        name: str,
        search: str,
        cron_schedule: str = "*/15 * * * *",
        alert_threshold: str = "5",
        action_webhook: bool = False,
        action_script: bool = False,
        action_email: bool = False,
        disabled: bool = True,
    ):
        owner = CONFIG["splunk"]["owner"]
        app = CONFIG["splunk"]["app"]
        url = f"{self.base_url}/servicesNS/{owner}/{app}/saved/searches"
        payload = {
            "name": name,
            "search": search if search.strip().startswith("search ") else f"search {search}",
            "cron_schedule": cron_schedule,
            "is_scheduled": "1",
            "disabled": "1" if disabled else "0",
            "alert_type": "number of events",
            "alert_comparator": "greater than",
            "alert_threshold": alert_threshold,
            "actions": ",".join(
                action
                for action, enabled in [
                    ("email", action_email),
                    ("webhook", action_webhook),
                    ("script", action_script),
                ]
                if enabled
            ),
            "output_mode": "json",
        }
        response = self.session.post(url, data=payload, timeout=30)
        response.raise_for_status()
        try:
            return response.json()
        except ValueError:
            return {"status_code": response.status_code, "text": response.text}

splunk = SplunkClient(
    base_url=CONFIG["splunk"]["mgmt_base_url"],
    verify_ssl=CONFIG["splunk"]["mgmt_verify_ssl"],
)
print("Splunk REST client is ready.")

## SPL Fundamentals — six live queries

Each query runs against the `citi_telemetry` index using the 500 events we just pushed.

We will demonstrate:

1. `stats count by severity`
2. `eval ... case(...)`
3. `timechart span=1h count by severity`
4. `top 10 endpoint_id by alert count`
5. `transaction endpoint_id maxspan=10m`
6. `rex` extraction from `message`

In [ ]:
spl_queries = {
    "stats_count_by_severity": '''
index=citi_telemetry source="postgres.alerts"
| stats count by severity
| sort - count
''',
    "eval_severity_score": '''
index=citi_telemetry source="postgres.alerts"
| eval severity_score=case(severity=="CRITICAL",4, severity=="HIGH",3, severity=="MEDIUM",2, true(),1)
| stats count as alert_count max(severity_score) as severity_score by severity
| sort - severity_score
''',
    "timechart_hourly_by_severity": '''
index=citi_telemetry source="postgres.alerts"
| timechart span=1h count by severity
''',
    "top_10_endpoint_ids": '''
index=citi_telemetry source="postgres.alerts"
| stats count as alert_count by endpoint_id
| sort - alert_count
| head 10
''',
    "transaction_endpoint_10m": '''
index=citi_telemetry source="postgres.alerts"
| sort 0 endpoint_id _time
| transaction endpoint_id maxspan=10m
| table endpoint_id eventcount duration severity message
| sort - eventcount
| head 20
''',
    "rex_error_type": '''
index=citi_telemetry source="postgres.alerts"
| rex field=message "(?<error_type>\w+\s\w+)"
| stats count by error_type
| sort - count
| head 20
''',
}

query_results = {}
for name, query in spl_queries.items():
    print(f"Running: {name}")
    df = splunk.run_search(query, earliest_time="-7d@d", latest_time="now", count=1000)
    query_results[name] = df
    print(f"Returned {len(df):,} rows")

In [ ]:
for name, df in query_results.items():
    display(Markdown(f"### {name}"))
    display(df.head(20))

## Search Commands Cheat Sheet

In [ ]:
cheat_sheet_df = pd.DataFrame(
    [
        {
            "command": "stats",
            "purpose": "Aggregate events into metrics and grouped summaries.",
            "example": 'index=citi_telemetry | stats count by severity',
        },
        {
            "command": "eval",
            "purpose": "Create or transform fields inline.",
            "example": '... | eval severity_score=case(severity=="CRITICAL",4,severity=="HIGH",3,severity=="MEDIUM",2,true(),1)',
        },
        {
            "command": "transaction",
            "purpose": "Group related events into a single logical incident.",
            "example": '... | transaction endpoint_id maxspan=10m',
        },
        {
            "command": "rex",
            "purpose": "Extract structured fields from raw text using regex.",
            "example": '... | rex field=message "(?<error_type>\\w+\\s\\w+)"',
        },
        {
            "command": "timechart",
            "purpose": "Build time-series views for trends and spikes.",
            "example": '... | timechart span=1h count by severity',
        },
        {
            "command": "top / sort / head",
            "purpose": "Rank entities such as noisy endpoints or top offenders.",
            "example": '... | stats count by endpoint_id | sort - count | head 10',
        },
    ]
)
display(cheat_sheet_df)

### Same cheat sheet as markdown

| command | purpose | example |
|---|---|---|
| `stats` | Aggregate events into metrics and grouped summaries. | `index=citi_telemetry | stats count by severity` |
| `eval` | Create or transform fields inline. | `... | eval severity_score=case(severity=="CRITICAL",4,severity=="HIGH",3,severity=="MEDIUM",2,true(),1)` |
| `transaction` | Group related events into a single logical incident. | `... | transaction endpoint_id maxspan=10m` |
| `rex` | Extract structured fields from raw text using regex. | `... | rex field=message "(?<error_type>\w+\s\w+)"` |
| `timechart` | Build time-series views for trends and spikes. | `... | timechart span=1h count by severity` |
| `top / sort / head` | Rank entities such as noisy endpoints or top offenders. | `... | stats count by endpoint_id | sort - count | head 10` |

## Alert Configuration

Operationally, a **saved search** is where SPL becomes action:

- **Saved search**: persistent search object
- **Threshold**: when to trigger
- **Action types**:
  - **email**
  - **webhook**
  - **script**

Typical example:

- Search every 15 minutes
- Trigger if CRITICAL alerts exceed 5
- Notify an operations team via email or webhook

In [ ]:
critical_alert_search = '''
index=citi_telemetry source="postgres.alerts" severity="CRITICAL"
| stats count as critical_alerts
'''

saved_search_payload_preview = {
    "name": "citi_critical_alert_threshold_demo",
    "search": critical_alert_search.strip(),
    "cron_schedule": "*/15 * * * *",
    "alert_type": "number of events",
    "alert_comparator": "greater than",
    "alert_threshold": "5",
    "actions": "webhook",
}
display(saved_search_payload_preview)

In [ ]:
create_saved_search_demo = False  # Set to True only when you want to create the saved search in Splunk.

if create_saved_search_demo:
    create_result = splunk.create_saved_search(
        name="citi_critical_alert_threshold_demo",
        search=critical_alert_search,
        cron_schedule="*/15 * * * *",
        alert_threshold="5",
        action_webhook=True,
        action_script=False,
        action_email=False,
        disabled=True,
    )
    print("Saved search created.")
    display(create_result)
else:
    print("Saved search creation skipped by design. Flip create_saved_search_demo=True to execute the REST POST.")

### REST API pattern for alert creation

**Endpoint**

`POST https://localhost:8089/servicesNS/admin/search/saved/searches`

**Key fields**

- `name`
- `search`
- `cron_schedule`
- `is_scheduled`
- `alert_type`
- `alert_comparator`
- `alert_threshold`
- `actions`

This is how Splunk alerting becomes part of a CI/CD-style operational automation flow.

## Dashboard Pattern

Classic Splunk dashboards are XML-based. A common pattern is:

- `<form>` root
- `<label>` for dashboard title
- `<row>` containers
- `<panel>` blocks
- `<chart>` or `<table>` visualization
- Inline `<search>` with `<query>`

### Sample panel XML — alert count by severity over time

In [ ]:
dashboard_panel_xml = '''
<panel>
  <title>Alert Count by Severity Over Time</title>
  <chart>
    <search>
      <query>
        index=citi_telemetry source="postgres.alerts"
        | timechart span=1h count by severity
      </query>
      <earliest>-24h@h</earliest>
      <latest>now</latest>
    </search>
    <option name="charting.chart">line</option>
    <option name="charting.legend.placement">right</option>
  </chart>
</panel>
'''.strip()

print(dashboard_panel_xml)

## What Just Happened

We exercised the six core SPL ideas with a Citi-style telemetry storyline:

1. **`stats`** answered: *How many alerts do we have by severity?*
2. **`eval`** answered: *Can we convert business severity into a numeric score for ranking and policy logic?*
3. **`timechart`** answered: *Are incidents clustering over time?*
4. **`stats + sort + head`** answered: *Which endpoints are the noisiest right now?*
5. **`transaction`** answered: *Which sequences of alerts belong to the same operational episode?*
6. **`rex`** answered: *Can we mine structure out of semi-structured messages without re-ingesting the data?*

### Citi framing

A Citi SOC analyst uses these exact command families to triage **6,000+ endpoints** across latency, throughput, and error-rate signals.

**SPL is the language of operational intelligence.**

In [ ]:
summary_df = pd.DataFrame(
    [
        {"command": "stats", "operational_question": "How many alerts by severity?"},
        {"command": "eval", "operational_question": "How do we convert text severity into a numeric decision signal?"},
        {"command": "timechart", "operational_question": "When do spikes happen over time?"},
        {"command": "top / sort / head", "operational_question": "Which endpoints are hottest?"},
        {"command": "transaction", "operational_question": "Which alerts belong to the same incident chain?"},
        {"command": "rex", "operational_question": "Which patterns can we extract from free-text messages?"},
    ]
)
display(summary_df)